In [ ]:
!pip install razdel

In [ ]:
import re
import json
import random
import joblib
import numpy as np

from datasets import load_dataset
from razdel import sentenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline

# НАСТРОЙКИ

DATASET_NAME = "iitolstykh/LLMTrace_classification"

MAX_TRAIN_DOCUMENTS = 100_000
MAX_VAL_DOCUMENTS = 20_000
MAX_TEST_DOCUMENTS = 20_000

MAX_FEATURES = 200_000
NGRAM_RANGE = (1, 2)

RANDOM_SEED = 42
MODEL_PATH = "logreg_sentence_detector.joblib"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ПОДГОТОВКА ДАННЫХ

def find_column(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


def normalize_label(value):
    """метки human=0 и ai=1"""
    value = str(value).strip().lower()

    if value=="human":
        return 0

    if value=="ai":
        return 1


def split_into_sentences(text):
    """Разбивает текст на предложения"""
    if not isinstance(text, str):
        return []

    text = text.strip()
    if not text:
        return []

    sentences = [
        item.text.strip()
        for item in sentenize(text)
        if item.text.strip()
    ]
    return [s for s in sentences if len(s) >= 10]


def documents_to_sentences(dataset, text_col, label_col, limit=None):
    """
    Сначала берём документы из готового split датасета,
    затем разбиваем каждый документ на предложения.
    Все предложения наследуют метку документа.
    """
    if limit is not None:
        dataset = dataset.select(range(min(limit, len(dataset))))

    texts = []
    labels = []

    for row in dataset:
        text = row[text_col]
        label = normalize_label(row[label_col])

        for sentence in split_into_sentences(text):
            texts.append(sentence)
            labels.append(label)

    return texts, np.asarray(labels, dtype=np.int32)


def main():
    print("Загружаем датасет...")
    dataset = load_dataset(DATASET_NAME)

    print("Доступные splits:", list(dataset.keys()))
    print("Столбцы:", dataset["train"].column_names)

    columns = dataset["train"].column_names

    text_col = "text"
    label_col = "label"

    print("\nПреобразуем документы в предложения...")

    X_train, y_train = documents_to_sentences(
        dataset["train"],
        text_col,
        label_col,
        MAX_TRAIN_DOCUMENTS,
    )

    val_split = "validation"

    if val_split is not None:
        X_val, y_val = documents_to_sentences(
            dataset[val_split],
            text_col,
            label_col,
            MAX_VAL_DOCUMENTS,
        )
    else:
        X_val, y_val = [], np.asarray([], dtype=np.int32)

    X_test, y_test = documents_to_sentences(
        dataset["test"],
        text_col,
        label_col,
        MAX_TEST_DOCUMENTS,
    )

    if len(X_train) == 0 or len(X_test) == 0:
        raise ValueError("После сегментации не осталось предложений.")

    print(f"Train: {len(X_train):,} предложений")
    print(f"Validation: {len(X_val):,} предложений")
    print(f"Test: {len(X_test):,} предложений")

    print("\nРаспределение классов:")
    print("Train:", dict(zip(*np.unique(y_train, return_counts=True))))
    print("Test:", dict(zip(*np.unique(y_test, return_counts=True))))

    # TF-IDF + LOGISTIC REGRESSION

    print("\nОбучаем TF-IDF + Logistic Regression...")

    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                max_features=MAX_FEATURES,
                ngram_range=NGRAM_RANGE,
                sublinear_tf=True,
                min_df=2,
                max_df=0.98,
                strip_accents=None,
                dtype=np.float32,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                C=4.0,
                max_iter=1000,
                class_weight="balanced",
                solver="liblinear",
                random_state=RANDOM_SEED,
            ),
        ),
    ])

    model.fit(X_train, y_train)

    # ОЦЕНКА

    print("\nОцениваем на test split...")
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision_ai": float(
            precision_score(y_test, y_pred, zero_division=0)
        ),
        "recall_ai": float(
            recall_score(y_test, y_pred, zero_division=0)
        ),
        "f1_ai": float(
            f1_score(y_test, y_pred, zero_division=0)
        ),
        "roc_auc": float(roc_auc_score(y_test, y_prob)),
    }

    print("\nTEST METRICS")
    for name, value in metrics.items():
        print(f"{name}: {value:.4f}")

    print("\nClassification report")
    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["human", "ai"],
        zero_division=0,
    ))

    print("Confusion matrix")
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

    with open("logreg_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    # Сохраняем весь pipeline: TF-IDF + классификатор.
    joblib.dump(model, MODEL_PATH)

    print(f"\nМодель сохранена: {MODEL_PATH}")
    print("Метрики сохранены: logreg_metrics.json")

if __name__ == "__main__":
    main()


Загружаем датасет...


README.md:   0%|          | 0.00/4.37k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 1.40GB            

train.jsonl: downloading bytes:           |  0.00B            

valid.jsonl: reconstructing file:   0%|          |  0.00B /  292MB            

...

test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/411440 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/86696 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/90950 [00:00<?, ? examples/s]

Доступные splits: ['train', 'validation', 'test']
Столбцы: ['lang', 'label', 'model', 'data_type', 'prompt_type', 'topic_id', 'text', 'prompt']

Преобразуем документы в предложения...
Train: 1,355,294 предложений
Validation: 257,975 предложений
Test: 285,707 предложений

Распределение классов:
Train: {np.int32(0): np.int64(745660), np.int32(1): np.int64(609634)}
Test: {np.int32(0): np.int64(165726), np.int32(1): np.int64(119981)}

Обучаем TF-IDF + Logistic Regression...

Оцениваем на test split...

=== TEST METRICS ===
accuracy: 0.7730
precision_ai: 0.7308
recall_ai: 0.7274
f1_ai: 0.7291
roc_auc: 0.8425

=== Classification report ===
              precision    recall  f1-score   support

       human       0.80      0.81      0.80    165726
          ai       0.73      0.73      0.73    119981

    accuracy                           0.77    285707
   macro avg       0.77      0.77      0.77    285707
weighted avg       0.77      0.77      0.77    285707

=== Confusion matrix ===
[[1335